In [2]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

In [3]:
# vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)
vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)


In [5]:
vix

,open,high,low,close,adjclose,volume,ticker
2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0,^VIX
2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0,^VIX
2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0,^VIX
2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0,^VIX
2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0,^VIX
...,...,...,...,...,...,...,...
2021-09-22,22.719999,23.400000,20.750000,20.870001,20.870001,0,^VIX
2021-09-23,19.910000,20.209999,18.420000,18.629999,18.629999,0,^VIX
2021-09-24,19.330000,20.410000,17.629999,17.750000,17.750000,0,^VIX
2021-09-27,17.780001,19.320000,17.740000,18.760000,18.760000,0,^VIX


In [7]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# from backtesting.test import SMA, GOOG, ^VIX


class SmaCross(Strategy):
    n1 = 10
    n2 = 20
    print(vix)
    def init(self):
        close = self.data.Close
        self.sma1 = self.I(vix, close, self.n1)
        self.sma2 = self.I(vix, close, self.n2)

    def next(self):
        if crossover(self.sma1, self.sma2):
            self.buy()
        elif crossover(self.sma2, self.sma1):
            self.sell()

vix.rename(columns={"open":"Open", "high":"High", "low":"Low","close":"Close"},inplace=True)
bt = Backtest(vix, SmaCross,
              cash=10000, commission=.002,
              exclusive_orders=True)

output = bt.run()
bt.plot()

                 open       high        low      close   adjclose  volume  \
2019-09-25  17.049999  18.450001  15.690000  15.960000  15.960000       0   
2019-09-26  16.230000  17.090000  15.350000  16.070000  16.070000       0   
2019-09-27  15.770000  18.690001  15.150000  17.219999  17.219999       0   
2019-09-30  17.230000  17.350000  16.200001  16.240000  16.240000       0   
2019-10-01  16.020000  18.620001  15.790000  18.559999  18.559999       0   
...               ...        ...        ...        ...        ...     ...   
2021-09-22  22.719999  23.400000  20.750000  20.870001  20.870001       0   
2021-09-23  19.910000  20.209999  18.420000  18.629999  18.629999       0   
2021-09-24  19.330000  20.410000  17.629999  17.750000  17.750000       0   
2021-09-27  17.780001  19.320000  17.740000  18.760000  18.760000       0   
2021-09-28  19.740000  24.820000  19.709999  23.250000  23.250000       0   

           ticker  
2019-09-25   ^VIX  
2019-09-26   ^VIX  
2019-09-27   ^V

RuntimeError: Indicator "df(C,10)" errored with exception: 'DataFrame' object is not callable

In [ ]:
output

In [ ]:
stats['_trades']

In [ ]:
from yahoo_fin.stock_info import get_data

df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d")

In [ ]:
df_sp500

In [ ]:
import backtrader as bt
import datetime

from matplotlib.dates import (HOURS_PER_DAY, MIN_PER_HOUR, SEC_PER_MIN,
                              MONTHS_PER_YEAR, DAYS_PER_WEEK,
                              SEC_PER_HOUR, SEC_PER_DAY,
                              num2date, rrulewrapper, YearLocator,
                              MicrosecondLocator)
from matplotlib import warnings


class TestStrategy(bt.Strategy):

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close
        self.order = None


    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log('BUY EXECUTED {}'.format(order.executed.price))
            elif order.issell():
                self.log('SELL EXECUTED {}'.format(order.executed.price))
        
            self.bar_executed = len(self)

        self.order = None


    def next(self):
        # Simply log the closing price of the series from the reference
        self.log('Close, %.2f' % self.dataclose[0])

        if self.order:
            return

        if not self.position:
            if self.dataclose[0] < self.dataclose[-1]:
                # current close less than previous close

                if self.dataclose[-1] < self.dataclose[-2]:
                    # previous close less than the previous close

                    # BUY, BUY, BUY!!! (with all possible default parameters)
                    self.log('BUY CREATE, %.2f' % self.dataclose[0])
                    self.order = self.buy()
        else:
            if len(self) >= (self.bar_executed + 5):
                self.log('SELL CREATED {}'.format(self.dataclose[0]))
                self.order = self.sell()
                

                
class Trader:

    cerebro = bt.Cerebro()

    cerebro.broker.set_cash(1000000)


    data = bt.feeds.YahooFinanceCSVData(dataname="/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/AMZN.csv",
                                        fromdate=datetime.datetime(2020, 1, 1),
                                        todate=datetime.datetime(2020, 9, 23),
                                        reverse=False)
    cerebro.adddata(data)

    cerebro.addstrategy(TestStrategy)
    cerebro.addsizer(bt.sizers.FixedSize, stake=300)

    print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

    cerebro.run()

    print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

    #Save output
    
    figure = cerebro.plot(style ='candlebars')[0][0]
    figure.savefig('result.png')